In [14]:
# Model download + local smoke test (SmolDocling)
# This will download weights the first time you run it (internet required).
# Goal: verify the model runs locally and produces some DocTags output.

import torch
from PIL import Image
from transformers import AutoModelForVision2Seq, AutoProcessor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "docling-project/SmolDocling-256M-preview"

# Test with a real image (fallback to a blank page if missing)
TEST_IMAGE_PATH = r"E:\school\BrainInk-Local\test2.jpeg"
LONGEST_EDGE = 768

try:
    dummy_img = Image.open(TEST_IMAGE_PATH).convert("RGB")
    print("Loaded test image:", TEST_IMAGE_PATH, "size=", dummy_img.size)
except Exception as e:
    print("Could not load test image; falling back to blank page:", type(e).__name__, str(e))
    dummy_img = Image.new("RGB", (768, 1024), color=(255, 255, 255))

processor = AutoProcessor.from_pretrained(MODEL_ID)
torch_dtype = torch.bfloat16 if DEVICE == "cuda" else torch.float32

attn_impl = "flash_attention_2" if DEVICE == "cuda" else "eager"
try:
    model = AutoModelForVision2Seq.from_pretrained(
        MODEL_ID,
        torch_dtype=torch_dtype,
        _attn_implementation=attn_impl,
    )
except Exception as e:
    # flash_attention_2 often isn't available on Windows; fall back safely.
    print("Attention impl fallback -> eager due to:", type(e).__name__)
    model = AutoModelForVision2Seq.from_pretrained(
        MODEL_ID,
        torch_dtype=torch_dtype,
        _attn_implementation="eager",
    )

model = model.to(DEVICE)
model.eval()

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "Convert this page to docling."},
        ],
    }
]

prompt = processor.apply_chat_template(messages, add_generation_prompt=True)

# For this model/processor combo on this image size, the vision encoder yields 144 image tokens.
# If you change image pre-processing, you may need to adjust this value.
IMAGE_SEQ_LEN = 144

inputs = processor(
    text=prompt,
    images=[[dummy_img]],
    return_tensors="pt",
    image_seq_len=IMAGE_SEQ_LEN,
    images_kwargs={
        "size": {"longest_edge": LONGEST_EDGE},
        "max_image_size": {"longest_edge": LONGEST_EDGE},
        "do_image_splitting": False,
    },
 )
inputs = inputs.to(DEVICE)

with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=256)

prompt_len = inputs.input_ids.shape[1]
trimmed = generated_ids[:, prompt_len:]
doctags = processor.batch_decode(trimmed, skip_special_tokens=False)[0].lstrip()

print("Using device:", DEVICE)
print("DocTags chars:", len(doctags))
print("DocTags preview:", doctags[:300])

Loaded test image: E:\school\BrainInk-Local\test2.jpeg size= (960, 1280)


Some kwargs in processor config are unused and will not have any effect: image_seq_len. 
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


Using device: cpu
DocTags chars: 507
DocTags preview: <doctag><picture><loc_1><loc_0><loc_500><loc_250><other></picture>
<text><loc_10><loc_250><loc_19><loc_255>-</text>
<text><loc_10><loc_255><loc_19><loc_260>-</text>
<text><loc_10><loc_260><loc_19><loc_265>-</text>
<text><loc_10><loc_265><loc_19><loc_270>-</text>
<text><loc_10><loc_270><loc_19><loc_2


# Vision/document conversion module (SmolDocling)
This notebook contains the **SmolDocling** side of the workflow.

**Contract**
- Input: handwritten answer image
- Output: **DocTags** (Docling representation) and/or derived **structured summary** string

SmolDocling is used for image understanding + document conversion (OCR/layout/formulas).
Reasoning/feedback generation is delegated to SmolLM2 in `smol_lm.ipynb`.

In [16]:
import re
from typing import Dict, List, Tuple

import torch
from PIL import Image, ImageDraw
from transformers import AutoModelForVision2Seq, AutoProcessor


def load_smoldocling(
    model_id: str = "docling-project/SmolDocling-256M-preview",
) -> Tuple[AutoProcessor, AutoModelForVision2Seq, str]:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    processor = AutoProcessor.from_pretrained(model_id)
    torch_dtype = torch.bfloat16 if device == "cuda" else torch.float32

    attn_impl = "flash_attention_2" if device == "cuda" else "eager"
    try:
        model = AutoModelForVision2Seq.from_pretrained(
            model_id,
            torch_dtype=torch_dtype,
            _attn_implementation=attn_impl,
        )
    except Exception as e:
        # flash_attention_2 often isn't available on Windows; fall back safely.
        print("Attention impl fallback -> eager due to:", type(e).__name__)
        model = AutoModelForVision2Seq.from_pretrained(
            model_id,
            torch_dtype=torch_dtype,
            _attn_implementation="eager",
        )

    model = model.to(device)
    model.eval()
    return processor, model, device


@torch.no_grad()
def convert_page_to_doctags(
    image: Image.Image,
    instruction: str = "Convert this page to docling.",
    model_id: str = "docling-project/SmolDocling-256M-preview",
    max_new_tokens: int = 2048,
    longest_edge: int = 768,
    image_seq_len: int = 144,
    ) -> str:
    """Convert an image to DocTags with a safe, explicit resize config."""
    processor, model, device = load_smoldocling(model_id=model_id)

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": instruction},
            ],
        }
    ]

    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(
        text=prompt,
        images=[[image.convert("RGB")]],
        return_tensors="pt",
        image_seq_len=int(image_seq_len),
        images_kwargs={
            "size": {"longest_edge": int(longest_edge)},
            "max_image_size": {"longest_edge": int(longest_edge)},
            "do_image_splitting": False,
        },
    ).to(device)
    generated_ids = model.generate(**inputs, max_new_tokens=int(max_new_tokens))

    prompt_length = inputs.input_ids.shape[1]
    trimmed_generated_ids = generated_ids[:, prompt_length:]
    doctags = processor.batch_decode(trimmed_generated_ids, skip_special_tokens=False)[0].lstrip()
    return doctags


def _parse_first_loc_bbox(tag_text: str) -> Tuple[int, int, int, int] | None:
    """Extract the first 4 <loc_…> integers as (x1,y1,x2,y2)."""
    locs = [int(x) for x in re.findall(r"<loc_(\d+)>", tag_text)[:4]]
    if len(locs) != 4:
        return None
    return locs[0], locs[1], locs[2], locs[3]


def parse_doctags_text_elements(doctags: str) -> List[Dict[str, object]]:
    """Parse all <text …>…</text> blocks from DocTags and keep bbox + content."""
    items: List[Dict[str, object]] = []
    # Matches blocks like: <text><loc_...><loc_...>Hello</text> (including newlines)
    for match in re.finditer(r"<text>(.*?)</text>", doctags, flags=re.DOTALL | re.IGNORECASE):
        inner = match.group(1)
        bbox = _parse_first_loc_bbox(inner)
        # Remove all <...> tags to keep only the visible text content
        content = re.sub(r"<[^>]+>", "", inner)
        content = re.sub(r"\s+", " ", content).strip()
        if not content:
            continue
        items.append({"type": "text", "bbox": bbox, "text": content})
    return items


def draw_bboxes_on_image(image: Image.Image, items: List[Dict[str, object]], limit: int = 200) -> Image.Image:
    """Overlay bounding boxes for parsed items onto a copy of the image."""
    img = image.convert("RGB").copy()
    draw = ImageDraw.Draw(img)
    for item in items[: int(limit)]:
        bbox = item.get("bbox")
        if not bbox:
            continue
        x1, y1, x2, y2 = bbox
        # Note: loc coords are in Docling coordinate space; this still gives a useful overlay for inspection.
        draw.rectangle([x1, y1, x2, y2], outline=(255, 0, 0), width=2)
    return img


# --- Demo: extract EVERYTHING we can from the picture ---
TEST_IMAGE_PATH = r"E:\school\BrainInk-Local\test2.jpeg"
try:
    demo_img = Image.open(TEST_IMAGE_PATH).convert("RGB")
    print("Loaded test image:", TEST_IMAGE_PATH, "size=", demo_img.size)
except Exception as e:
    raise FileNotFoundError(f"Could not load image at {TEST_IMAGE_PATH}: {e}")

# 1) Convert to DocTags (full output)
demo_doctags = convert_page_to_doctags(
    demo_img,
    max_new_tokens=2048,
    longest_edge=768,
    image_seq_len=144,
)

print("\n=== FULL DOCTAGS (raw) ===\n")
print(demo_doctags)

# 2) Parse ALL text elements + print them all
items = parse_doctags_text_elements(demo_doctags)
print("\n=== ALL EXTRACTED TEXT ELEMENTS ===\n")
print("Count:", len(items))
for i, it in enumerate(items, start=1):
    bbox = it.get("bbox")
    bbox_str = "None" if bbox is None else f"{bbox}"
    print(f"{i:03d}. bbox={bbox_str} | {it['text']}")

# 3) Optional: overlay bounding boxes on the image so you can visually inspect coverage
try:
    import matplotlib.pyplot as plt

    overlay = draw_bboxes_on_image(demo_img, items, limit=250)
    plt.figure(figsize=(8, 10))
    plt.imshow(overlay)
    plt.axis("off")
    plt.title("DocTags text bboxes overlay")
    plt.show()
except Exception as e:
    print("(Overlay skipped)", type(e).__name__, str(e))

Loaded test image: E:\school\BrainInk-Local\test2.jpeg size= (960, 1280)


Some kwargs in processor config are unused and will not have any effect: image_seq_len. 
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.



=== FULL DOCTAGS (raw) ===

<doctag><picture><loc_1><loc_0><loc_500><loc_250><other></picture>
<text><loc_10><loc_250><loc_19><loc_255>-</text>
<text><loc_10><loc_255><loc_19><loc_260>-</text>
<text><loc_10><loc_260><loc_19><loc_265>-</text>
<text><loc_10><loc_265><loc_19><loc_270>-</text>
<text><loc_10><loc_270><loc_19><loc_275>-</text>
<text><loc_10><loc_275><loc_19><loc_280>-</text>
<text><loc_10><loc_280><loc_19><loc_285>-</text>
<text><loc_10><loc_285><loc_19><loc_290>-</text>
<text><loc_10><loc_290><loc_19><loc_295>-</text>
<text><loc_10><loc_295><loc_19><loc_300>-</text>
<text><loc_10><loc_300><loc_19><loc_305>-</text>
<text><loc_10><loc_305><loc_19><loc_310>-</text>
<text><loc_10><loc_310><loc_19><loc_315>-</text>
<text><loc_10><loc_315><loc_19><loc_320>-</text>
<text><loc_10><loc_320><loc_19><loc_325>-</text>
<text><loc_10><loc_325><loc_19><loc_330>-</text>
<text><loc_10><loc_330><loc_19><loc_335>-</text>
<text><loc_10><loc_335><loc_19><loc_340>-</text>
<text><loc_10><loc_340